<a href="https://colab.research.google.com/github/danielferber/PUC2026/blob/main/TrabalhoFinal.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Titulo
teste

### Mount Google Drive

First, we need to mount your Google Drive to access the files. Follow the instructions after running the next cell to authenticate.

In [9]:
from google.colab import drive
drive.mount('/content/drive', readonly=True)

Mounted at /content/drive


### Unzip the Training Dataset

Extracts training dataset zip file available in google drive and extract it into /content/GTSRB_training/dataset. Extract only files relevant for training.

In [11]:
import zipfile
import os
import shutil

def extract_gtsrb_zip(zip_file_path, output_dir):
    """
    Extracts the GTSRB dataset from a zip file to the specified output directory.
    Assumes the structure within the zip is 'GTSRB/Final_Training/Images/'.
    """
    # Ensure the target output directory is clean before extracting
    if os.path.exists(output_dir):
        shutil.rmtree(output_dir)
    os.makedirs(output_dir, exist_ok=True)

    with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
        for member in zip_ref.namelist():
            # We are interested in files under 'GTSRB/Final_Training/Images/'
            if member.startswith('GTSRB/Final_Training/Images/') and not member.endswith('/'):
                # Extract the part after 'GTSRB/Final_Training/Images/'
                # e.g., '00000/00000_00000.ppm'
                extracted_path_suffix = member[len('GTSRB/Final_Training/Images/'):]

                if extracted_path_suffix: # Ensure it's not an empty string
                    target_file_path = os.path.join(output_dir, extracted_path_suffix)

                    # Ensure the directory for the file exists
                    os.makedirs(os.path.dirname(target_file_path), exist_ok=True)

                    # Extract the file
                    with zip_ref.open(member) as source, open(target_file_path, "wb") as target:
                        target.write(source.read())
            elif member.startswith('GTSRB/') and 'Final_Training/Images/' not in member and not member.endswith('/'):
                # Also extract other files directly under 'GTSRB/' if they are not part of the image structure
                relative_path = member[len('GTSRB/'):]
                if relative_path and not relative_path.startswith('Final_Training'):
                    target_file_path = os.path.join(output_dir, relative_path)
                    os.makedirs(os.path.dirname(target_file_path), exist_ok=True)
                    with zip_ref.open(member) as source, open(target_file_path, "wb") as target:
                        target.write(source.read())

    print(f"Dataset unzipped to: {output_dir}")
    print(os.listdir(output_dir))


# IMPORTANT: Update this path to the correct location of your zip file in Google Drive
zip_file_path = '/content/drive/My Drive/PUC/PAI/GTSRB_Final_Training_Images.zip' # This is an example path, adjust as needed
training_output_dir = 'GTSRB_training_dataset'

extract_gtsrb_zip(zip_file_path, training_output_dir)


Extracting training dataset from /content/drive/My Drive/PUC/PAI/GTSRB_Final_Training_Images.zip...
Dataset unzipped to: GTSRB_training_dataset
['00031', '00040', '00021', '00020', '00001', '00029', '00039', '00035', '00030', '00027', '00041', 'Readme-Images.txt', '00018', '00015', '00008', '00033', '00004', '00042', '00023', '00036', '00032', '00014', '00019', '00017', '00034', '00005', '00012', '00028', '00000', '00006', '00025', '00011', '00016', '00009', '00022', '00026', '00007', '00003', '00038', '00013', '00037', '00024', '00002', '00010']


### Load and Preprocess the GTSRB Dataset

First, we need to load the images from the unzipped dataset and preprocess them for training. This involves reading images, resizing them to a uniform size, and normalizing pixel values.

In [21]:
import tensorflow as tf
import numpy as np
import os
import cv2
from sklearn.model_selection import train_test_split
from tensorflow.keras.utils import to_categorical

# Define constants
IMG_HEIGHT = 30 # Reverting to 30x30 to match previous execution output
IMG_WIDTH = 30 # Reverting to 30x30 to match previous execution output
NUM_CLASSES = 43 # GTSRB has 43 classes
DATA_DIR = 'GTSRB_training_dataset'
BATCH_SIZE = 64

# Function to collect image paths and their corresponding labels
def collect_image_paths_and_labels(data_dir, num_classes):
    image_paths = []
    labels = []
    for item in os.listdir(data_dir):
        if os.path.isdir(os.path.join(data_dir, item)) and item.isdigit():
            class_id = int(item)
            if class_id < num_classes:
                path = os.path.join(data_dir, item)
                for img_name in os.listdir(path):
                    if img_name.endswith(('.ppm', '.png', '.jpg')):
                        image_paths.append(os.path.join(path, img_name))
                        labels.append(class_id)
    return np.array(image_paths), np.array(labels)

print("Collecting image paths and labels...")
all_image_paths, all_labels = collect_image_paths_and_labels(DATA_DIR, NUM_CLASSES)
print(f"Collected {len(all_image_paths)} image paths with {len(np.unique(all_labels))} classes.")

# Split data into training and validation sets (paths and integer labels)
X_train_paths, X_val_paths, y_train_labels, y_val_labels = train_test_split(
    all_image_paths, all_labels, test_size=0.2, random_state=42, stratify=all_labels
)

# Helper function to read image using OpenCV, as tf.image.decode_image does not support .ppm
def _read_image_cv2(image_path_tensor):
    image_path = image_path_tensor.numpy().decode('utf-8')
    img = cv2.imread(image_path)
    if img is None:
        # It's crucial to handle cases where imread fails. For tf.py_function,
        # returning a tensor of zeros or raising an error can be options.
        # For now, let's raise an error to indicate a problem.
        raise ValueError(f"Failed to read image: {image_path}")
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB) # Convert BGR to RGB
    return img

# Function to load, preprocess, and decode an image
def preprocess_image(image_path, label):
    # Use tf.py_function to wrap cv2.imread
    img = tf.py_function(func=_read_image_cv2, inp=[image_path], Tout=tf.uint8)
    img.set_shape([IMG_HEIGHT, IMG_WIDTH, 3]) # Set shape for consistency
    img = tf.image.resize(img, [IMG_HEIGHT, IMG_WIDTH]) # Resize again to ensure correct shape after py_function
    img = img / 255.0  # Normalize to [0, 1]
    label = tf.one_hot(label, NUM_CLASSES) # One-hot encode the label
    return img, label

# Create tf.data.Dataset for training
train_dataset = tf.data.Dataset.from_tensor_slices((X_train_paths, y_train_labels))
train_dataset = train_dataset.map(preprocess_image, num_parallel_calls=tf.data.AUTOTUNE)
train_dataset = train_dataset.shuffle(buffer_size=1024).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

# Create tf.data.Dataset for validation
val_dataset = tf.data.Dataset.from_tensor_slices((X_val_paths, y_val_labels))
val_dataset = val_dataset.map(preprocess_image, num_parallel_calls=tf.data.AUTOTUNE)
val_dataset = val_dataset.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

print(f"Training dataset created with {tf.data.experimental.cardinality(train_dataset).numpy() * BATCH_SIZE} elements (approx).")
print(f"Validation dataset created with {tf.data.experimental.cardinality(val_dataset).numpy() * BATCH_SIZE} elements (approx).")

Collected 39209 image paths with 43 classes.
Training dataset created with 31424 elements (approx).
Validation dataset created with 7872 elements (approx).


### Build a Simple Convolutional Neural Network (CNN)

Next, we'll define a basic CNN model using Keras. This model will consist of a few convolutional layers, pooling layers, and a dense output layer. The `input_shape` will now correctly reflect the `IMG_HEIGHT` and `IMG_WIDTH` defined in the previous step.

In [22]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout

def build_model():
    model = Sequential([
        Conv2D(32, (3, 3), activation='relu', input_shape=(IMG_HEIGHT, IMG_WIDTH, 3)),
        MaxPooling2D((2, 2)),
        Conv2D(64, (3, 3), activation='relu'),
        MaxPooling2D((2, 2)),
        Flatten(),
        Dense(128, activation='relu'),
        Dropout(0.5),
        Dense(NUM_CLASSES, activation='softmax')
    ])
    return model

model = build_model()
model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_8 (Conv2D)               │ (None, 28, 28, 32)     │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_8 (MaxPooling2D)  │ (None, 14, 14, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_9 (Conv2D)               │ (None, 12, 12, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_9 (MaxPooling2D)  │ (None, 6, 6, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_4 (Flatten)             │ (None, 2304)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 128)            │       295,040 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ (None, 43)             │         5,547 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 319,979 (1.22 MB)

 Trainable params: 319,979 (1.22 MB)

 Non-trainable params: 0 (0.00 B)

### Compile and Train the Model

Now, we'll compile the model by specifying an optimizer, loss function, and metrics. Then, we'll train the model using the preprocessed training data from the `tf.data.Dataset`.

In [ ]:
model.compile(optimizer='adam',
              loss='categorical_crossentropy',
              metrics=['accuracy'])

# Train the model using the tf.data.Dataset objects
EPOCHS = 10

print("Training the model...")
history = model.fit(train_dataset,
                    epochs=EPOCHS,
                    validation_data=val_dataset)

print("Model training complete.")

Training the model...
Epoch 1/10
491/491 ━━━━━━━━━━━━━━━━━━━━ 50s 99ms/step - accuracy: 0.3726 - loss: 2.2740 - val_accuracy: 0.7516 - val_loss: 0.9332
Epoch 2/10
241/491 ━━━━━━━━━━━━━━━━━━━━ 23s 96ms/step - accuracy: 0.6164 - loss: 1.2312

### Evaluate the Model

Finally, let's evaluate the trained model on the validation set to see its performance, using the `val_dataset`.

In [7]:
loss, accuracy = model.evaluate(val_dataset, verbose=0)
print(f"Validation Loss: {loss:.4f}")
print(f"Validation Accuracy: {accuracy:.4f}")

NameError: name 'X_val' is not defined